# Dissertation
Sebastián Miramontes Soto

DATA SET

In [ ]:
import os
import re
import time
import requests
import numpy as np
import pandas as pd

# SETTINGS

API_KEY = ""

BASE_URL = "https://v3.football.api-sports.io"

OUTPUT_FOLDER = "scouting_data"

REQUEST_DELAY = 0.4

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

session = requests.Session()

session.headers.update({"x-apisports-key": API_KEY})


# LEAGUES

LEAGUES = [
    {"country": "spain", "league": "La Liga", "id": 140,"seasons": [2025]},
    {"country": "england", "league": "Premier League", "id": 39, "seasons": [2025]},
    {"country": "italy", "league": "Serie A", "id": 135, "seasons": [2025]},
    {"country": "france", "league": "Ligue 1", "id": 61, "seasons": [2025]},
    {"country": "germany", "league": "Bundesliga", "id": 78, "seasons": [2025]},
    {"country": "portugal", "league": "Primeira Liga", "id": 94, "seasons": [2025]},
    {"country": "netherlands", "league": "Eredivisie", "id": 88, "seasons": [2025]},
    {"country": "turkey", "league": "Super Lig", "id": 203, "seasons": [2025]},
    {"country": "mexico", "league": "Liga MX", "id": 262, "seasons": [2025]},
    {"country": "argentina", "league": "Primera Division", "id": 128, "seasons": [2025, 2026]},
    {"country": "brazil", "league": "Serie A Brazil", "id": 71, "seasons": [2025, 2026]},
    {"country": "usa", "league": "MLS", "id": 253, "seasons": [2025, 2026]}]

# API REQUEST

def api_request(endpoint, params=None):
    response = session.get(f"{BASE_URL}/{endpoint}", params=params, timeout=60)

    response.raise_for_status()

    data = response.json()

    if data.get("errors"):
        raise RuntimeError(f"API error in {endpoint}: {data['errors']}")

    return data

# FINISHED FIXTURES

def get_finished_fixtures(league, season):
    data = api_request("fixtures", {"league": league["id"], "season": season})

    fixtures = []

    for item in data["response"]:
        fixture = item["fixture"]

        if fixture["status"]["short"] not in ["FT", "AET", "PEN"]:
            continue

        fixtures.append({"country": league["country"], "league": league["league"], "league_id": league["id"], "season": season, "fixture_id": fixture["id"], "fixture_date": fixture["date"]})

    return pd.DataFrame(fixtures)

# PLAYER STATISTICS FOR ONE FIXTURE

def get_fixture_players(fixture):
    data = api_request("fixtures/players", {"fixture": fixture["fixture_id"]})

    rows = []

    for team_data in data["response"]:
        team = team_data["team"]

        for player_data in team_data["players"]:
            player = player_data["player"]

            for stats in player_data["statistics"]:
                games = stats.get("games") or {}
                shots = stats.get("shots") or {}
                goals = stats.get("goals") or {}
                passes = stats.get("passes") or {}
                tackles = stats.get("tackles") or {}
                duels = stats.get("duels") or {}
                dribbles = stats.get("dribbles") or {}
                fouls = stats.get("fouls") or {}
                cards = stats.get("cards") or {}
                penalty = stats.get("penalty") or {}

                rows.append({
                    "country": fixture["country"],
                    "league": fixture["league"],
                    "league_id": fixture["league_id"],
                    "season": fixture["season"],
                    "fixture_id": fixture["fixture_id"],
                    "fixture_date": fixture["fixture_date"],

                    "team_id": team["id"],
                    "team_name": team["name"],

                    "player_id": player["id"],
                    "player_name": player["name"],

                    "position": games.get("position"),
                    "minutes": games.get("minutes"),
                    "substitute": games.get("substitute"),
                    "rating": games.get("rating"),

                    "offsides": stats.get("offsides"),

                    "shots_total": shots.get("total"),
                    "shots_on": shots.get("on"),

                    "goals_total": goals.get("total"),
                    "assists": goals.get("assists"),

                    "passes_total": passes.get("total"),
                    "passes_key": passes.get("key"),
                    "passes_accuracy": passes.get("accuracy"),

                    "tackles_total": tackles.get("total"),
                    "blocks": tackles.get("blocks"),
                    "interceptions": tackles.get("interceptions"),

                    "duels_total": duels.get("total"),
                    "duels_won": duels.get("won"),

                    "dribbles_attempts": dribbles.get("attempts"),
                    "dribbles_success": dribbles.get("success"),
                    "dribbles_past": dribbles.get("past"),

                    "fouls_drawn": fouls.get("drawn"),
                    "fouls_committed": fouls.get("committed"),

                    "yellow_cards": cards.get("yellow"),
                    "yellowred_cards": cards.get("yellowred"),
                    "red_cards": cards.get("red"),

                    "penalty_won": penalty.get("won"),
                    "penalty_committed": penalty.get("commited"),
                    "penalty_scored": penalty.get("scored"),
                    "penalty_missed": penalty.get("missed")})

    return rows

# DOWNLOAD ALL MATCH DATA

def download_match_data(league, season):
    fixtures = get_finished_fixtures(league, season)

    rows = []

    for _, fixture in fixtures.iterrows():
        rows.extend(get_fixture_players(fixture))

        time.sleep(REQUEST_DELAY)

    return pd.DataFrame(rows)

# PLAYER PROFILES

def download_player_profiles(league, season):
    page = 1
    rows = []

    while True:
        data = api_request(
            "players",
            {"league": league["id"], "season": season, "page": page})

        for item in data["response"]:
            player = item["player"]

            rows.append({
                "player_id": player["id"],
                "age": player.get("age"),
                "nationality": player.get("nationality"),
                "height": player.get("height"),
                "weight": player.get("weight")})

        total_pages = data["paging"]["total"]

        if page >= total_pages:
            break

        page += 1

        time.sleep(REQUEST_DELAY)

    profiles = pd.DataFrame(rows)

    return profiles.drop_duplicates(subset="player_id")

# HELPER FUNCTIONS

def extract_number(value):
    if pd.isna(value):
        return np.nan

    match = re.search(r"\d+(?:\.\d+)?", str(value))

    if match:
        return float(match.group())

    return np.nan


def most_common_value(series):
    mode = series.dropna().mode()

    if len(mode) > 0:
        return mode.iloc[0]

    return np.nan


def join_unique_values(series):
    values = (series.dropna().astype(str).unique())

    return " / ".join(sorted(values))

# AGGREGATE ONE SEASON

def aggregate_season_players(match_data, profiles):
    numeric_columns = [
        "minutes",
        "rating",
        "passes_accuracy",
        "offsides",
        "shots_total",
        "shots_on",
        "goals_total",
        "assists",
        "passes_total",
        "passes_key",
        "tackles_total",
        "blocks",
        "interceptions",
        "duels_total",
        "duels_won",
        "dribbles_attempts",
        "dribbles_success",
        "dribbles_past",
        "fouls_drawn",
        "fouls_committed",
        "yellow_cards",
        "yellowred_cards",
        "red_cards",
        "penalty_won",
        "penalty_committed",
        "penalty_scored",
        "penalty_missed"]

    for column in numeric_columns:
        match_data[column] = pd.to_numeric(match_data[column], errors="coerce")

    match_data["minutes"] = (match_data["minutes"].fillna(0))

    count_columns = [
        column
        for column in numeric_columns if column not in ["rating", "passes_accuracy"]]

    match_data[count_columns] = (match_data[count_columns].fillna(0))

    match_data["appearance"] = (match_data["minutes"] > 0).astype(int)

    match_data["start"] = ((match_data["minutes"] > 0) & (match_data["substitute"] == False)).astype(int)

    match_data["rated_minutes"] = np.where(match_data["rating"].notna(), match_data["minutes"], 0)

    match_data["rating_minutes"] = np.where(match_data["rating"].notna(), (match_data["rating"] * match_data["minutes"]), 0)

    match_data["pass_weight"] = np.where(match_data["passes_accuracy"].notna() & (match_data["passes_total"] > 0), match_data["passes_total"], 0)

    match_data["pass_accuracy_weight"] = np.where(
        match_data["passes_accuracy"].notna() & (match_data["passes_total"] > 0), (match_data["passes_accuracy"] * match_data["passes_total"]), 0)

    sum_columns = [
        "appearance",
        "start",
        "minutes",
        "rated_minutes",
        "rating_minutes",
        "pass_weight",
        "pass_accuracy_weight",
        "offsides",
        "shots_total",
        "shots_on",
        "goals_total",
        "assists",
        "passes_total",
        "passes_key",
        "tackles_total",
        "blocks",
        "interceptions",
        "duels_total",
        "duels_won",
        "dribbles_attempts",
        "dribbles_success",
        "dribbles_past",
        "fouls_drawn",
        "fouls_committed",
        "yellow_cards",
        "yellowred_cards",
        "red_cards",
        "penalty_won",
        "penalty_committed",
        "penalty_scored",
        "penalty_missed"]

    aggregations = {column: "sum" for column in sum_columns}

    aggregations.update({"player_name": "first", "position": most_common_value})

    players = (match_data.groupby(["country", "league", "season", "player_id"], dropna=False).agg(aggregations).reset_index())

    players = players.rename(columns={"appearance": "appearances", "start": "starts"})

    players = players.merge(profiles, on="player_id", how="left")

    players["height"] = (players["height"].apply(extract_number))

    players["weight"] = (players["weight"].apply(extract_number))

    return players

# COMBINE ALL PERIODS INTO ONE ROW PER PLAYER

def combine_player_periods(players):
    sum_columns = [
        "appearances",
        "starts",
        "minutes",
        "rated_minutes",
        "rating_minutes",
        "pass_weight",
        "pass_accuracy_weight",
        "offsides",
        "shots_total",
        "shots_on",
        "goals_total",
        "assists",
        "passes_total",
        "passes_key",
        "tackles_total",
        "blocks",
        "interceptions",
        "duels_total",
        "duels_won",
        "dribbles_attempts",
        "dribbles_success",
        "dribbles_past",
        "fouls_drawn",
        "fouls_committed",
        "yellow_cards",
        "yellowred_cards",
        "red_cards",
        "penalty_won",
        "penalty_committed",
        "penalty_scored",
        "penalty_missed"]

    aggregations = {column: "sum" for column in sum_columns}

    aggregations.update({
        "player_name": "first",
        "country": join_unique_values,
        "league": join_unique_values,
        "season": join_unique_values,
        "age": "max",
        "position": most_common_value,
        "nationality": most_common_value,
        "height": most_common_value,
        "weight": most_common_value})

    combined = (players.groupby("player_id", dropna=False).agg(aggregations).reset_index())

    combined["rating_weighted"] = np.where(combined["rated_minutes"] > 0,
        (combined["rating_minutes"] / combined["rated_minutes"]), np.nan)

    combined["passes_accuracy_pct"] = np.where(combined["pass_weight"] > 0,
        (combined["pass_accuracy_weight"] / combined["pass_weight"]), np.nan)

    combined["goals_plus_assists"] = (combined["goals_total"] + combined["assists"])

    combined["non_penalty_goals"] = (combined["goals_total"] - combined["penalty_scored"])

    combined["tackles_plus_interceptions"] = (combined["tackles_total"] + combined["interceptions"])

    per90_columns = [
        "offsides",
        "shots_total",
        "shots_on",
        "goals_total",
        "assists",
        "goals_plus_assists",
        "non_penalty_goals",
        "passes_total",
        "passes_key",
        "tackles_total",
        "blocks",
        "interceptions",
        "tackles_plus_interceptions",
        "duels_total",
        "duels_won",
        "dribbles_attempts",
        "dribbles_success",
        "dribbles_past",
        "fouls_drawn",
        "fouls_committed",
        "yellow_cards",
        "yellowred_cards",
        "red_cards"]

    for column in per90_columns:
        combined[f"{column}_per90"] = np.where(combined["minutes"] > 0, (combined[column] / combined["minutes"] * 90), np.nan)

    combined["shots_on_target_pct"] = np.where(combined["shots_total"] > 0, (combined["shots_on"] / combined["shots_total"] * 100), np.nan)

    combined["duels_won_pct"] = np.where(combined["duels_total"] > 0, (combined["duels_won"] / combined["duels_total"] * 100), np.nan)

    combined["dribbles_success_pct"] = np.where(combined["dribbles_attempts"] > 0, (combined["dribbles_success"] / combined["dribbles_attempts"] * 100), np.nan)

    combined = combined.drop(columns=["rated_minutes", "rating_minutes", "pass_weight", "pass_accuracy_weight"], errors="ignore")

    combined = combined.replace([np.inf, -np.inf], np.nan)

    return combined

# FINAL COLUMN FORMAT

def select_final_columns(players):
    qualitative_columns = [
        "country",
        "league",
        "season",
        "player_name",
        "age",
        "position",
        "nationality",
        "height",
        "weight"]

    basic_columns = [
        "appearances",
        "starts",
        "minutes",
        "goals_total",
        "assists",
        "penalty_won",
        "penalty_committed",
        "penalty_scored",
        "penalty_missed",
        "yellow_cards",
        "yellowred_cards",
        "red_cards"]

    per90_columns = [
        column
        for column in players.columns
        if column.endswith("_per90")]

    percentage_columns = [
        column
        for column in players.columns
        if column.endswith("_pct")]

    final_columns = (
        qualitative_columns
        + basic_columns
        + per90_columns
        + percentage_columns
        + ["rating_weighted"])

    return players[final_columns]

# PROCESS ONE LEAGUE-SEASON

def process_season(league, season):
    match_data = download_match_data(league, season)

    profiles = download_player_profiles(league, season)

    return aggregate_season_players(match_data, profiles)

# PROCESS ALL COUNTRIES

all_player_periods = []

for league in LEAGUES:
    country_periods = []

    for season in league["seasons"]:
        season_players = process_season( league, season)

        country_periods.append(season_players)

        all_player_periods.append(season_players)

    country_periods = pd.concat(country_periods, ignore_index=True)

    country_players = combine_player_periods(country_periods)

    country_dataset = select_final_columns(country_players)

    country_dataset = (country_dataset.sort_values("player_name").reset_index(drop=True))

    country_file = os.path.join(OUTPUT_FOLDER, f"{league['country']}_dataset.csv")

    country_dataset.to_csv(country_file, index=False)

    print(f"READY {league['country']}_dataset.csv")

# FINAL SCOUTING DATASET

all_player_periods = pd.concat(all_player_periods, ignore_index=True)

all_players_combined = combine_player_periods(all_player_periods)

scouting_dataset = select_final_columns(all_players_combined)

scouting_dataset = (scouting_dataset.sort_values("player_name").reset_index(drop=True))

final_file = os.path.join(OUTPUT_FOLDER,"scouting_dataset.csv")

scouting_dataset.to_csv(final_file,index=False)

print("\nSCOUTING DATASET:",scouting_dataset.shape)

pd.set_option("display.max_columns", None)

display(scouting_dataset.head(20))

READY spain_dataset.csv
READY england_dataset.csv
READY italy_dataset.csv
READY france_dataset.csv
READY germany_dataset.csv
READY portugal_dataset.csv
READY netherlands_dataset.csv
READY turkey_dataset.csv
READY mexico_dataset.csv
READY argentina_dataset.csv
READY brazil_dataset.csv
READY usa_dataset.csv

SCOUTING DATASET: (9572, 49)


,country,league,season,player_name,age,position,nationality,height,weight,appearances,starts,minutes,goals_total,assists,penalty_won,penalty_committed,penalty_scored,penalty_missed,yellow_cards,yellowred_cards,red_cards,offsides_per90,shots_total_per90,shots_on_per90,goals_total_per90,assists_per90,goals_plus_assists_per90,non_penalty_goals_per90,passes_total_per90,passes_key_per90,tackles_total_per90,blocks_per90,interceptions_per90,tackles_plus_interceptions_per90,duels_total_per90,duels_won_per90,dribbles_attempts_per90,dribbles_success_per90,dribbles_past_per90,fouls_drawn_per90,fouls_committed_per90,yellow_cards_per90,yellowred_cards_per90,red_cards_per90,passes_accuracy_pct,shots_on_target_pct,duels_won_pct,dribbles_success_pct,rating_weighted
0,usa,MLS,2025,AJ Marcucci,26.0,G,USA,191.0,86.0,2,2,180.0,0.0,0.0,0.0,0.0,0,0,0,0.0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,29.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,18.152542,NaN,NaN,NaN,6.500000
1,turkey,Super Lig,2025,Aaron Appindangoyé,33.0,D,Gabon,185.0,85.0,4,4,315.0,0.0,0.0,0.0,0.0,0,0,0,0.0,0,0.285714,0.285714,0.285714,0.000000,0.000000,0.000000,0.000000,39.714286,0.000000,0.285714,1.714286,1.714286,2.000000,8.571429,2.857143,0.000000,0.000000,0.000000,0.285714,1.428571,0.000000,0.0,0.000000,29.359712,100.000000,33.333333,NaN,6.285714
2,netherlands,Eredivisie,2025,Aaron Bouwman,18.0,D,Netherlands,188.0,71.0,10,8,770.0,1.0,0.0,0.0,0.0,0,0,2,0.0,0,0.000000,0.701299,0.467532,0.116883,0.000000,0.116883,0.116883,66.155844,0.116883,1.168831,0.584416,0.818182,1.987013,5.376623,2.454545,0.233766,0.233766,0.350649,0.000000,1.168831,0.233766,0.0,0.000000,67.358657,66.666667,45.652174,100.000000,7.051948
3,usa,MLS,2025 / 2026,Aaron Herrera,28.0,D,Guatemala,180.0,71.0,33,28,2459.0,0.0,2.0,0.0,0.0,0,0,11,0.0,2,0.036600,0.512403,0.146401,0.000000,0.073200,0.073200,0.000000,37.332249,1.537210,1.573810,0.475803,1.171208,2.745018,8.271655,4.245628,1.573810,0.732005,0.439203,0.549004,1.207808,0.402603,0.0,0.073200,32.444118,28.571429,51.327434,46.511628,6.921431
4,england,Premier League,2025,Aaron Hickey,23.0,D,Scotland,178.0,72.0,21,8,716.0,0.0,0.0,0.0,0.0,0,0,1,0.0,0,0.000000,0.628492,0.125698,0.000000,0.000000,0.000000,0.000000,35.824022,0.000000,1.885475,0.754190,1.005587,2.891061,8.296089,4.148045,0.879888,0.000000,0.754190,1.131285,0.879888,0.125698,0.0,0.000000,21.375439,20.000000,50.000000,0.000000,6.633147
5,usa,MLS,2025 / 2026,Aaron Long,33.0,D,USA,186.0,79.0,20,20,1627.0,1.0,0.0,0.0,0.0,0,0,1,0.0,0,0.000000,0.221266,0.110633,0.055317,0.000000,0.055317,0.055317,71.358328,0.221266,1.161647,0.553165,0.331899,1.493546,5.421020,3.650891,0.110633,0.000000,0.276583,0.221266,0.719115,0.055317,0.0,0.000000,72.006977,50.000000,67.346939,0.000000,7.085618
6,england / spain,La Liga / Premier League,2025,Aaron Mayol,22.0,D,England,181.0,NaN,0,0,0.0,0.0,0.0,0.0,0.0,0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,netherlands,Eredivisie,2025,Aaron Meijers,38.0,D,Netherlands,176.0,77.0,12,5,511.0,0.0,0.0,0.0,0.0,0,0,0,0.0,0,0.000000,0.176125,0.176125,0.000000,0.000000,0.000000,0.000000,32.935421,0.176125,1.937378,0.176125,0.000000,1.937378,8.277886,4.227006,0.880626,0.352250,1.585127,1.232877,0.880626,0.000000,0.0,0.000000,18.636364,100.000000,51.063830,40.000000,6.511546
8,argentina,Primera Division,2025 / 2026,Aaron Molinas,25.0,M,Argentina,175.0,67.0,42,40,3266.0,5.0,7.0,0.0,0.0,3,0,7,0.0,0,0.027557,0.854256,0.440906,0.137783,0.192897,0.330680,0.055113,54.066136,1.956522,1.322719,0.192897,0.606246,1.928965,6.586038,3.196571,1.184936,0.799143,0.303123,0.826699,0.633803,0.192897,0.0,0.000000,46.504077,51.612903,48.535565,67.441860,7.171617
9,turkey,Super Lig,2025,Aaron Opoku,26.0,D,Germany,185.0,71.0,19,14,1235.0,0.0,2.0,0.0,0.0,0,0,3,0.0,0,0.145749,0.510121,0.218623,0.000000,0.145749,0.145749,0.00